In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Cargar los datos
csv_path = r"C:\Users\japje\Documents\B.G\wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)
texts = df["chunk_text"].tolist()
id_to_text = df[["chunk_id", "chunk_text"]].reset_index(drop=True)

# Embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
dim = embeddings.shape[1]

# FAISS index
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# TinyLLaMA fallback
tinyllama_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_id)
model = AutoModelForCausalLM.from_pretrained(tinyllama_model_id)
local_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)

# OpenAI setup
openai_api_key = "OPENAI_API_KEY"  # Reemplaza con tu clave real
client = OpenAI(api_key=openai_api_key)

def ask_question(query, k=5):
    """
    Responde una pregunta usando RAG con OpenAI y TinyLLaMA como fallback.
    Args:
        query (str): Pregunta del usuario.
        k (int): Número de chunks a recuperar.
    Returns:
        dict: {'answer': str, 'chunks': list, 'model': str}
    """
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)
    retrieved_chunks = [id_to_text.iloc[idx]["chunk_text"] for idx in indices[0]]
    context = "\n".join(retrieved_chunks[:3])
    max_context_len = 1500
    context = context[:max_context_len]
    prompt = f"""You are a helpful assistant. Use the following context to answer the question:

Context:
{context}

Question: {query}
Answer:"""

    answer = None
    model_used = None

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=500
        )
        answer = response.choices[0].message.content
        model_used = "OpenAI GPT-3.5"
    except Exception as e:
        print(f"⚠️ OpenAI failed: {e}")
        output = local_pipe(prompt, max_new_tokens=300, temperature=0.7, do_sample=True)
        answer = output[0]["generated_text"].split("Answer:")[-1].strip()
        model_used = "TinyLLaMA"

    return {
        "answer": answer,
        "chunks": retrieved_chunks,
        "model": model_used
    }

# Ejemplo de uso:
if __name__ == "__main__":
    result = ask_question("who is bob brown", k=3)
    print("\n🔎 Retrieved Chunks:\n")
    for chunk in result["chunks"]:
        print(f"- {chunk}\n---")
    print(f"\n💬 Final Answer (via {result['model']}):\n{result['answer']}")